# Task 4 — Preprocessing & Indexing

**Mata Kuliah:** Temu Kembali Informasi  
**Dataset:** 50 putusan Pidana Khusus Narkotika (hasil Task 2 & 3)  

Notebook ini menerapkan pipeline NLP (case folding, cleaning, stopword removal,
stemming Sastrawi), membangun bobot **TF-IDF**, serta **inverted index** sebagai
dasar mesin temu kembali.

## 0. Instalasi Dependensi

In [ ]:
!pip install -q Sastrawi==1.0.1 scikit-learn==1.4.1.post1 pandas==2.2.1

## 1. Ambil Repositori & Impor Modul

In [ ]:
REPO_URL = "https://github.com/<username>/Dataset-Narkotika_XXX_YYY.git"
REPO_DIR = "Dataset-Narkotika_XXX_YYY"

import os, sys

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL

sys.path.insert(0, os.path.abspath(REPO_DIR))
os.chdir(REPO_DIR)

## 2. Muat Data Hasil Ekstraksi

In [ ]:
import pandas as pd

df = pd.read_csv("data/extracted_decisions.csv")
print("Jumlah dokumen :", len(df))
df.head()

## 3. Pipeline Preprocessing

Kolom teks yang diproses: `Amar Putusan` (gabungkan dengan `Barang Bukti` bila perlu).

In [ ]:
from src.preprocessing import preprocess_corpus, tokenize

raw_corpus = df["Amar Putusan"].fillna("").astype(str).tolist()
processed_corpus = preprocess_corpus(raw_corpus)

print("=== Contoh SEBELUM ===")
print(raw_corpus[0][:300])
print("\n=== Contoh SESUDAH ===")
print(processed_corpus[0][:300])

## 4. Pembobotan TF-IDF

In [ ]:
from src.preprocessing import build_tfidf_index

vectorizer, tfidf_matrix, feature_names = build_tfidf_index(processed_corpus)

print("Bentuk matriks TF-IDF :", tfidf_matrix.shape)
print("Jumlah term unik       :", len(feature_names))
print("Contoh term            :", list(feature_names[:15]))

## 5. Inverted Index

In [ ]:
from src.preprocessing import build_inverted_index

inverted_index = build_inverted_index(processed_corpus)
print("Jumlah term di inverted index :", len(inverted_index))

term = "narkotika"
print(f"Dokumen yang memuat '{term}':", sorted(inverted_index.get(term, [])))

## 6. Uji Query Sederhana (Cosine Similarity)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

query = "pidana penjara narkotika"
query_vec = vectorizer.transform([query])
scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

top_idx = scores.argsort()[::-1][:5]
for rank, idx in enumerate(top_idx, 1):
    print(f"{rank}. doc#{idx} | skor={scores[idx]:.4f} | {df.loc[idx, 'No Putusan']}")

## 7. Simpan Hasil Preprocessing

In [ ]:
df["Teks_Bersih"] = processed_corpus
df.to_csv("data/preprocessed_decisions.csv", index=False, encoding="utf-8")
print("Tersimpan: data/preprocessed_decisions.csv")